# research-note_3
### B1 → B3 → B2 → B4

In [3]:
import torch
from torchvision.datasets import MNIST

mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)

In [4]:
print(mnist_train.data.shape)
print(mnist_train.data.dtype)
print(mnist_train.targets.shape)
print(mnist_train.targets[:10])

torch.Size([60000, 28, 28])
torch.uint8
torch.Size([60000])
tensor([5, 0, 4, 1, 9, 2, 1, 3, 1, 4])


In [5]:
print(mnist_train.data.dtype)
print(mnist_train.data.min().item())
print(mnist_train.data.max().item())

torch.uint8
0
255


## 1 MNIST Data #B1
$$
x \in [0,1]^{784}, \qquad
y(x) = e_{\text{label}} \in \{0,1\}^{10}, \qquad
\text{e.g. label} = 6 \;\Rightarrow\; y(x) = (0,0,0,0,0,0,1,0,0,0)^{T}
$$
$$
\texttt{t.sum(dim=1)}[i] = \sum_{j} t_{ij}, \qquad
\texttt{t.sum(dim=0)}[j] = \sum_{i} t_{ij}, \qquad
\texttt{t.argmax(dim=1)}[i] = \arg\max_{j} \, t_{ij}
$$

In [6]:
X = mnist_train.data.reshape(-1, 784).float() / 255

X_train = X[:50000]
X_val = X[50000:]
labels_train = mnist_train.targets[:50000]
labels_val = mnist_train.targets[50000:]
Y_train = torch.eye(10)[labels_train]

X_test = mnist_test.data.reshape(-1, 784).float() / 255
labels_test = mnist_test.targets

print(X_train.shape, X_val.shape, X_test.shape)
print("X_train.max =", X_train.max().item(), "X_train.min =", X_train.min().item(), "X_train.dtype =", X_train.dtype)
print("X_val.max =", X_val.max().item(), "X_val.min =", X_val.min().item(), "X_val.dtype =", X_val.dtype)
print("X_test.max =", X_test.max().item(), "X_test.min =", X_test.min().item(), "X_test.dtype =", X_test.dtype)
print((Y_train.sum(dim = 1) == 1.0 ).all())
print((Y_train.argmax(dim = 1) == labels_train).all())

torch.Size([50000, 784]) torch.Size([10000, 784]) torch.Size([10000, 784])
X_train.max = 1.0 X_train.min = 0.0 X_train.dtype = torch.float32
X_val.max = 1.0 X_val.min = 0.0 X_val.dtype = torch.float32
X_test.max = 1.0 X_test.min = 0.0 X_test.dtype = torch.float32
tensor(True)
tensor(True)


## 2 SGD Update & Training Loop #B2
$$
a' = \sigma(w a + b), \qquad
C(w,b) = \frac{1}{2n} \sum_{x} \lVert y(x) - a \rVert^{2}
$$
$$
w_k \to w_k' = w_k - \frac{\eta}{m} \sum_{j=1}^{m} \frac{\partial C_{X_j}}{\partial w_k}, \qquad
b_l \to b_l' = b_l - \frac{\eta}{m} \sum_{j=1}^{m} \frac{\partial C_{X_j}}{\partial b_l}
$$
$$
\texttt{W.grad}[i, j] = \frac{\partial L}{\partial W_{ij}}, \qquad
\texttt{b.grad}[j] = \frac{\partial L}{\partial b_{j}}
$$
$$
W \leftarrow W - \eta \, \texttt{W.grad}, \qquad
b \leftarrow b - \eta \, \texttt{b.grad}
$$
$$
\texttt{cost(out, Y)} = \frac{1}{2m} \sum_{i=1}^{m} \sum_{j=1}^{10} \left( Y_{ij} - \texttt{out}_{ij} \right)^{2}
= \frac{1}{2m} \sum_{x \in \text{batch}} \lVert y(x) - a \rVert^{2}
$$

### Program Structure

| function | input | output |
|---|---|---|
| `feedforward(X, params)` | `X` (N, 784), `params` | (N, 10) |
| `evaluate(params, X, labels)` | `params`, `X` (N, 784), `labels` (N,) | int |
| `sgd_step(params, lr)` | `params`, `lr` | none (updates `params` in place) |
| `train(...)` | sizes, data, epochs, m, lr | trained `params` |

```text
train:
    build params (randn, requires_grad=True)
    repeat epochs times:
        shuffle data
        split into mini-batches of size m, for each batch:
            out = feedforward(X_batch, params)   → (m, 10)
            L   = cost(out, Y_batch)             → scalar
            L.backward()                         → grads into each W.grad, b.grad
            sgd_step(params, lr)                 → update, zero grads
        print evaluate(params, X_test, labels_test)
    return params
```

In [12]:
def sgd_step(params, lr):
    with torch.no_grad():
        for W, b in params:
            W -= lr * W.grad
            b -= lr * b.grad
            W.grad.zero_()
            b.grad.zero_()

def cost(out, Y):
    return ((Y - out) ** 2).sum() / (2 * len(Y))

def init_params(sizes):
    return [(torch.randn(n_in, n_out, requires_grad=True), torch.randn(n_out, requires_grad=True)) for n_in, n_out in zip(sizes[:-1], sizes[1:])]

#for W, b in init_params([784, 30, 10]):
#   print(W.shape, b.shape, W.requires_grad)

perm = torch.randperm(len(X_train))
X_shuf = X_train[perm]
Y_shuf = Y_train[perm]
for k in range(0, len(X_train), 10):
    X_batch = X_shuf[k:k+10]
    Y_batch = Y_shuf[k:k+10]
    print(X_batch.shape, Y_batch.shape)
    break


torch.Size([10, 784]) torch.Size([10, 10])


## 3 Evaluation #B3
$$
\text{correct} = \sum_{x \in \text{test}} \mathbf{1}\!\left[\, \arg\max_{j} a_j(x) = \text{label}(x) \,\right]
$$
$$
X' = \sigma(XW + b), \qquad
X \in \mathbb{R}^{N \times n_{\text{in}}}, \;
W \in \mathbb{R}^{n_{\text{in}} \times n_{\text{out}}}, \;
b \in \mathbb{R}^{n_{\text{out}}}, \qquad
W = w^{T}
$$
$$
(XW + b)_{ij} = \sum_{k} X_{ik} W_{kj} + b_{j}
$$

In [8]:
def feedforward(X, params):
    for W, b in params:
        X = torch.sigmoid(X @ W + b)
    return X

params = [
    (torch.randn(784, 30), torch.randn(30)),
    (torch.randn(30, 10), torch.randn(10))
]

def evaluate(params, X, labels):
    with torch.no_grad():
        out = feedforward(X, params)
        return (out.argmax(dim = 1) == labels).sum().item()

print("Train accuracy:", evaluate(params, X_train, labels_train) / len(X_train))

Train accuracy: 0.10426


## 4 Hyperparameters & 784-10 #B4
$$
\Delta v = -\eta \nabla C, \qquad
\Delta C \approx \nabla C \cdot \Delta v = -\eta \lVert \nabla C \rVert^{2}
$$

## 5 Baselines
$$
D(x) = \sum_{i=1}^{784} x_i, \qquad
\bar{D}_d = \frac{1}{\lvert S_d \rvert} \sum_{x \in S_d} D(x), \qquad
\hat{y}(x) = \arg\min_{d \in \{0,\dots,9\}} \bigl\lvert D(x) - \bar{D}_d \bigr\rvert
$$

## 6 Towards Deep Learning
$$
a^{L} = \sigma\!\left( w^{L} \, \sigma\!\left( w^{L-1} \cdots \sigma\!\left( w^{2} x + b^{2} \right) \cdots + b^{L-1} \right) + b^{L} \right)
$$
$$
f(x_1, \dots, x_n) = \sum_{q=0}^{2n} \Phi_q\!\left( \sum_{p=1}^{n} \phi_{q,p}(x_p) \right)
$$